# IAN 603 · Module 4
# Aggregating Data with PostgreSQL

Aggregation turns detailed rows into summaries.

The central question for this module is:

> **What should one row represent after the aggregation?**

That question determines your `GROUP BY`, aggregate functions, and validation strategy.

We will move from basic summaries to grouped summaries, conditional metrics, subtotals, and introductory window functions.

Course module:  
https://ali2449.github.io/IAN-603-Preparing-Data-for-Analytics-/s_module4.html


## What you will learn

By the end of this notebook, you should be able to:

1. Explain how aggregation changes row grain.
2. Distinguish `COUNT(*)`, `COUNT(column)`, and `COUNT(DISTINCT ...)`.
3. Use `SUM`, `AVG`, `MIN`, and `MAX`.
4. Explain how `NULL` affects aggregate functions.
5. Group by one or more columns.
6. Recognize and fix a common `GROUP BY` error.
7. Explain the difference between `WHERE` and `HAVING`.
8. Build conditional metrics with `CASE` and PostgreSQL `FILTER`.
9. Use `ROLLUP`, `CUBE`, and `GROUPING SETS`.
10. Explain how window functions differ from normal aggregation.
11. Rank rows and calculate group averages without collapsing detail.
12. Calculate a running total.
13. Validate grouped totals against source data.


## 0. Install required Python packages

Run this cell if the packages are not already installed.


In [ ]:
%pip install -q pandas sqlalchemy psycopg2-binary ipywidgets


## 1. Load the course database helper

This notebook uses a small course helper to open the PostgreSQL connection window and display query results.

**You only need this notebook file.** The helper is downloaded automatically when needed.

If you want your own copy:

[Download `ian603_database.py`](https://github.com/ali2449/IAN-603-Preparing-Data-for-Analytics-/raw/refs/heads/MSIA/ian603_database.py)


In [ ]:
from pathlib import Path
import urllib.request
from sqlalchemy import text

MODULE_FILE = Path("ian603_database.py")
MODULE_URL = (
    "https://raw.githubusercontent.com/"
    "ali2449/IAN-603-Preparing-Data-for-Analytics-/MSIA/"
    "ian603_database.py"
)

if not MODULE_FILE.exists():
    try:
        urllib.request.urlretrieve(MODULE_URL, MODULE_FILE)
        print("Course database helper downloaded.")
    except Exception:
        print("The database helper could not be downloaded automatically.")
        print("Use the download link above, then run this cell again.")
        raise

from ian603_database import IAN603Database

db = IAN603Database()
display_query = db.display_query
execute_query = db.execute_query
execute_script = db.execute_script

print("Database helper ready.")


## 2. Connect to PostgreSQL

Run the next cell to open the connection window.

| Setting | Default |
|---|---|
| Host | `localhost` |
| Database | `postgres` |
| Username | `postgres` |
| Port | `5432` |

Enter your PostgreSQL password and click **Connect**.


In [ ]:
db.show_connection_panel()


## 3. Check the connection

After clicking **Connect**, run this cell once.


In [ ]:
if db.connected:
    db.test_connection()
else:
    print("No active PostgreSQL connection.")
    print("Return to the connection panel above and click Connect.")


# Part I · Start with Grain

Before writing an aggregate query, identify what one source row represents.

| Table | One source row represents |
|---|---|
| `admissions` | one admission |
| `sales` | one sales transaction |
| `employees` | one employee |
| `product_sales` | one product sales summary row |
| `grades` | one student-course-semester grade |

Then state the desired output grain:

> **I want one output row per ________.**

Examples:

- one row for the whole table
- one row per diagnosis
- one row per subject + diagnosis
- one row per sales category
- one row per employee

This habit is more important than memorizing aggregate syntax.


## 4. Create or reset the Module 4 database

Re-run this cell whenever you want a clean starting point.


In [ ]:
setup_sql = "-- ============================================================\n-- IAN 603 - MODULE 4\n-- Aggregating Data with PostgreSQL\n-- ============================================================\n-- Companion file for s_module4.html\n-- PostgreSQL / pgAdmin Query Tool\n--\n-- Topics:\n-- COUNT, SUM, AVG, MIN, MAX, GROUP BY, HAVING, FILTER,\n-- CASE-based conditional aggregation, ROLLUP, CUBE,\n-- GROUPING SETS, and introductory window calculations.\n--\n-- Main habit: aggregation changes row grain. Define the desired\n-- grouping grain before writing the aggregate query.\n-- ============================================================\n\n-- ============================================================\n-- SECTION 1 - CREATE / RESET TEACHING DATABASE\n-- ============================================================\n\nDROP TABLE IF EXISTS grades,product_sales,employees,sales,admissions CASCADE;CREATE TABLE admissions(hadm_id integer PRIMARY KEY,subject_id integer NOT NULL,admit_time timestamp NOT NULL,discharge_time timestamp,diagnosis text NOT NULL);INSERT INTO admissions VALUES(1001,1,'2024-01-02 08:00','2024-01-05 12:00','Pneumonia'),(1002,1,'2024-03-10 09:00','2024-03-12 14:00','Fracture'),(1003,2,'2024-02-01 10:00','2024-02-04 10:00','Pneumonia'),(1004,2,'2024-04-18 12:00',NULL,'Pneumonia'),(1005,3,'2024-01-15 07:00','2024-01-16 18:00','Migraine'),(1006,3,'2024-05-06 11:00','2024-05-09 09:00','Fracture'),(1007,4,'2024-06-03 09:00','2024-06-08 09:00','Pneumonia'),(1008,5,'2024-06-14 13:00','2024-06-15 10:00','Asthma'),(1009,5,'2024-07-01 16:00','2024-07-04 11:00','Fracture'),(1010,6,'2024-07-20 06:00',NULL,'Pneumonia');CREATE TABLE sales(sale_id integer PRIMARY KEY,product text NOT NULL,quantity integer NOT NULL,price numeric(10,2) NOT NULL,category text NOT NULL,sale_date date NOT NULL);INSERT INTO sales VALUES(1,'Laptop',3,900,'Technology','2024-01-02'),(2,'Monitor',5,220,'Technology','2024-01-04'),(3,'Keyboard',6,45,'Technology','2024-01-05'),(4,'Desk',4,350,'Office','2024-01-08'),(5,'Chair',8,180,'Office','2024-01-09'),(6,'Notebook',12,8,'Office','2024-01-10'),(7,'Blender',5,95,'Home','2024-01-12'),(8,'Lamp',7,40,'Home','2024-01-14'),(9,'Storage Bin',10,22,'Home','2024-01-16'),(10,'Headphones',4,120,'Technology','2024-01-18');CREATE TABLE employees(employee_id integer PRIMARY KEY,name text NOT NULL,salary numeric(10,2) NOT NULL,department text NOT NULL,hire_date date NOT NULL);INSERT INTO employees VALUES(1,'Jordan Kim',92000,'Analytics','2021-04-12'),(2,'Taylor Brooks',81000,'Analytics','2022-07-01'),(3,'Casey Morgan',76000,'Analytics','2023-02-14'),(4,'Riley Chen',98000,'Engineering','2020-09-10'),(5,'Morgan Patel',88000,'Engineering','2022-11-20'),(6,'Jamie Rivera',72000,'Operations','2021-06-18'),(7,'Avery Scott',69000,'Operations','2023-03-25'),(8,'Cameron Bell',65000,'Operations','2024-01-15');CREATE TABLE product_sales(category text,product text,sales numeric(10,2));INSERT INTO product_sales VALUES('Technology','Laptop',2700),('Technology','Monitor',1100),('Technology','Keyboard',270),('Office','Desk',1400),('Office','Chair',1440),('Home','Blender',475),('Home','Lamp',280);CREATE TABLE grades(student_id integer,course text,grade numeric(5,2),semester text);INSERT INTO grades SELECT g,'IAN 603',72+(g%24),'Fall 2024' FROM generate_series(1,60) g;INSERT INTO grades SELECT 100+g,'IAN 601',70+(g%25),'Fall 2024' FROM generate_series(1,55) g;INSERT INTO grades SELECT 200+g,'IAN 620',74+(g%20),'Fall 2024' FROM generate_series(1,45) g;INSERT INTO grades SELECT 300+g,'IAN 603',75+(g%20),'Spring 2025' FROM generate_series(1,35) g;"

execute_script(setup_sql)


## 5. Inspect the detailed admissions first

The source grain is **one row per admission**.


In [ ]:
display_query("""
SELECT hadm_id, subject_id, admit_time, discharge_time, diagnosis
FROM admissions
ORDER BY hadm_id;
""", title="Admissions source rows")


## 6. Inspect the detailed sales rows

The source grain is **one row per sales transaction**.


In [ ]:
display_query("""
SELECT sale_id, product, category, quantity, price, sale_date
FROM sales
ORDER BY sale_id;
""", title="Sales source rows")


# Part II · Aggregate the Whole Table

Without `GROUP BY`, the entire filtered dataset is treated as one group.

The output grain becomes:

> **one row for the whole result**


## 7. Three meanings of COUNT

| Expression | What it counts |
|---|---|
| `COUNT(*)` | all rows |
| `COUNT(discharge_time)` | rows where discharge_time is not NULL |
| `COUNT(DISTINCT subject_id)` | unique non-NULL subjects |

These answer different questions even though they all use `COUNT`.


In [ ]:
display_query("""
SELECT
    COUNT(*) AS all_admissions,
    COUNT(discharge_time) AS completed_admissions,
    COUNT(DISTINCT subject_id) AS unique_patients
FROM admissions;
""", title="Three COUNT meanings")


## 8. SUM, AVG, MIN, and MAX

For sales data:

| Function | Example question |
|---|---|
| `SUM(quantity)` | How many units were sold? |
| `AVG(price)` | What is the average row price? |
| `MIN(price)` | What is the lowest price? |
| `MAX(price)` | What is the highest price? |
| `SUM(quantity * price)` | What is total revenue? |


In [ ]:
display_query("""
SELECT
    COUNT(*) AS sale_rows,
    SUM(quantity) AS units_sold,
    ROUND(AVG(price), 2) AS average_price,
    MIN(price) AS lowest_price,
    MAX(price) AS highest_price,
    ROUND(SUM(quantity * price), 2) AS total_revenue
FROM sales;
""", title="Overall sales summary")


# Part III · NULL and Aggregation

Most aggregate functions ignore `NULL`.

Suppose stay lengths are:

| admission | stay_days |
|---:|---:|
| A | 3 |
| B | 5 |
| C | NULL |

Then `AVG(stay_days)` is 4, not 2.67.

The missing row is excluded from the average rather than treated as zero.


## 9. Step 1 — calculate stay length for each completed admission


In [ ]:
display_query("""
SELECT
    hadm_id,
    subject_id,
    diagnosis,
    ROUND(
        (EXTRACT(EPOCH FROM (discharge_time - admit_time)) / 86400.0)::numeric,
        2
    ) AS stay_days
FROM admissions
WHERE discharge_time IS NOT NULL
ORDER BY hadm_id;
""", title="Completed stay lengths")


## 10. Step 2 — summarize the completed stays


In [ ]:
display_query("""
SELECT
    ROUND(
        AVG(EXTRACT(EPOCH FROM (discharge_time - admit_time)) / 86400.0)::numeric,
        2
    ) AS average_stay_days,
    COUNT(discharge_time) AS completed_admissions
FROM admissions
WHERE discharge_time IS NOT NULL;
""", title="Average completed length of stay")


# Part IV · GROUP BY Changes the Grain

Start with detailed rows:

| hadm_id | diagnosis |
|---:|---|
| 1001 | Pneumonia |
| 1002 | Fracture |
| 1003 | Pneumonia |
| 1004 | Pneumonia |

After grouping by diagnosis:

| diagnosis | admission_count |
|---|---:|
| Pneumonia | 3 |
| Fracture | 1 |

The individual admission rows disappear because the output grain becomes:

> **one row per diagnosis**


## 11. One row per diagnosis


In [ ]:
display_query("""
SELECT
    diagnosis,
    COUNT(*) AS admission_count
FROM admissions
GROUP BY diagnosis
ORDER BY admission_count DESC, diagnosis;
""", title="Admissions by diagnosis")


## 12. Group by more than one column

If we group by both `subject_id` and `diagnosis`, the output grain is:

> **one row per subject-diagnosis combination**


In [ ]:
display_query("""
SELECT
    subject_id,
    diagnosis,
    COUNT(*) AS admission_count
FROM admissions
GROUP BY subject_id, diagnosis
ORDER BY subject_id, diagnosis;
""", title="Admissions by subject and diagnosis")


# Part V · Learn from a GROUP BY Error

Consider this query:

    SELECT diagnosis, subject_id, COUNT(*)
    FROM admissions
    GROUP BY diagnosis;

The intended grain says **one row per diagnosis**, but `subject_id` may have several possible values inside each diagnosis group.

PostgreSQL refuses to guess which subject ID should be shown.


## 13. Run the incorrect query and read the error

This error is intentional.


In [ ]:
try:
    with db.engine.connect() as conn:
        conn.execute(text("""
            SELECT diagnosis, subject_id, COUNT(*) AS admission_count
            FROM admissions
            GROUP BY diagnosis;
        """))
except Exception as e:
    print("✅ Expected PostgreSQL grouping error")
    print()
    print(str(e).split("\n")[0])


### How to fix it

There are two valid directions:

**Change the grain:** group by both `diagnosis` and `subject_id`.

**Keep the grain:** remove `subject_id` from the output.

The correct choice depends on the question, not on the error message alone.


# Part VI · WHERE vs HAVING

These clauses filter at different stages.

| Clause | Filters |
|---|---|
| `WHERE` | source rows before grouping |
| `HAVING` | groups after aggregation |

A useful mental flow is:

**FROM → WHERE → GROUP BY → aggregates → HAVING → SELECT → ORDER BY**


## 14. Read a complete grouped query step by step

Question:

> For sales rows priced at least $25, which categories sold more than 10 total units?

- `WHERE price >= 25` removes source rows first.
- `GROUP BY category` creates category groups.
- `SUM(quantity)` summarizes each group.
- `HAVING SUM(quantity) > 10` removes small groups.


In [ ]:
display_query("""
SELECT
    category,
    SUM(quantity) AS units_sold,
    ROUND(SUM(quantity * price), 2) AS category_revenue
FROM sales
WHERE price >= 25
GROUP BY category
HAVING SUM(quantity) > 10
ORDER BY category_revenue DESC;
""", title="WHERE before grouping, HAVING after grouping")


## 15. HAVING without WHERE

Keep only diagnosis groups with more than one admission.


In [ ]:
display_query("""
SELECT
    diagnosis,
    COUNT(*) AS admission_count
FROM admissions
GROUP BY diagnosis
HAVING COUNT(*) > 1
ORDER BY admission_count DESC, diagnosis;
""", title="Diagnosis groups with more than one admission")


# Part VII · Conditional Aggregation

Conditional aggregation creates several metrics inside the same grouping grain.

Suppose a subject has:

| diagnosis |
|---|
| Pneumonia |
| Fracture |
| Pneumonia |

The desired row might be:

| subject | pneumonia | fracture |
|---:|---:|---:|
| 1 | 2 | 1 |

PostgreSQL gives us two useful styles: `CASE` and `FILTER`.


## 16. Conditional counts with CASE


In [ ]:
display_query("""
SELECT
    subject_id,
    SUM(
        CASE WHEN diagnosis = 'Pneumonia' THEN 1 ELSE 0 END
    ) AS pneumonia_admissions,
    SUM(
        CASE WHEN diagnosis = 'Fracture' THEN 1 ELSE 0 END
    ) AS fracture_admissions
FROM admissions
GROUP BY subject_id
ORDER BY subject_id;
""", title="Conditional counts with CASE")


### How CASE works inside SUM

For the Pneumonia column, each source row becomes either 1 or 0.

Then `SUM` adds those values inside each subject group.


## 17. The same conditional counts with FILTER


In [ ]:
display_query("""
SELECT
    subject_id,
    COUNT(*) FILTER (
        WHERE diagnosis = 'Pneumonia'
    ) AS pneumonia_admissions,
    COUNT(*) FILTER (
        WHERE diagnosis = 'Fracture'
    ) AS fracture_admissions
FROM admissions
GROUP BY subject_id
ORDER BY subject_id;
""", title="Conditional counts with FILTER")


## 18. CASE vs FILTER

| Pattern | Strength |
|---|---|
| `SUM(CASE ...)` | flexible and widely portable |
| `COUNT/SUM(...) FILTER (...)` | concise and readable in PostgreSQL |

The syntax matters less than knowing the grouping grain and the meaning of each metric.


# Part VIII · Advanced Grouping

Sometimes one report needs several aggregation levels.

For example:

- product detail
- category subtotal
- grand total

PostgreSQL supports `ROLLUP`, `CUBE`, and `GROUPING SETS`.


## 19. Inspect the product-sales source


In [ ]:
display_query("""
SELECT category, product, sales
FROM product_sales
ORDER BY category, product;
""", title="Product-sales source")


## 20. ROLLUP: hierarchical subtotals

`ROLLUP(category, product)` produces a hierarchy:

1. category + product detail
2. category subtotal
3. grand total

`GROUPING()` helps distinguish subtotal placeholders from real NULL values.


In [ ]:
display_query("""
SELECT
    CASE
        WHEN GROUPING(category) = 1 THEN 'All categories'
        ELSE category
    END AS category_label,
    CASE
        WHEN GROUPING(product) = 1 THEN 'All products'
        ELSE product
    END AS product_label,
    SUM(sales) AS total_sales,
    GROUPING(category, product) AS grouping_code
FROM product_sales
GROUP BY ROLLUP(category, product)
ORDER BY
    GROUPING(category),
    category,
    GROUPING(product),
    product;
""", title="ROLLUP")


## 21. CUBE: every grouping combination

`CUBE(category, product)` can produce:

- category + product
- category only
- product only
- grand total

Use it when all combinations are genuinely useful.


In [ ]:
display_query("""
SELECT
    CASE
        WHEN GROUPING(category) = 1 THEN 'All categories'
        ELSE category
    END AS category_label,
    CASE
        WHEN GROUPING(product) = 1 THEN 'All products'
        ELSE product
    END AS product_label,
    SUM(sales) AS total_sales,
    GROUPING(category, product) AS grouping_code
FROM product_sales
GROUP BY CUBE(category, product)
ORDER BY
    GROUPING(category),
    category,
    GROUPING(product),
    product;
""", title="CUBE")


## 22. GROUPING SETS: request exactly the levels you need

Here we ask only for:

- category totals
- product totals
- grand total


In [ ]:
display_query("""
SELECT
    CASE
        WHEN GROUPING(category) = 1 THEN 'All categories'
        ELSE category
    END AS category_label,
    CASE
        WHEN GROUPING(product) = 1 THEN 'All products'
        ELSE product
    END AS product_label,
    SUM(sales) AS total_sales
FROM product_sales
GROUP BY GROUPING SETS (
    (category),
    (product),
    ()
)
ORDER BY
    GROUPING(category),
    category,
    GROUPING(product),
    product;
""", title="GROUPING SETS")


## 23. Choose the simplest advanced grouping tool

| Tool | Best fit |
|---|---|
| `ROLLUP` | hierarchical subtotals |
| `CUBE` | every combination |
| `GROUPING SETS` | selected levels only |


# Part IX · GROUP BY vs Window Functions

Normal aggregation collapses detail rows.

A window function can keep each detail row while adding group-level information.


## 24. GROUP BY: one row per department


In [ ]:
display_query("""
SELECT
    department,
    ROUND(AVG(salary), 2) AS department_average
FROM employees
GROUP BY department
ORDER BY department;
""", title="GROUP BY department")


## 25. Window average: one row per employee


In [ ]:
display_query("""
SELECT
    name,
    department,
    salary,
    ROUND(
        AVG(salary) OVER (
            PARTITION BY department
        ),
        2
    ) AS department_average
FROM employees
ORDER BY department, salary DESC;
""", title="Department average without collapsing employees")


### Compare the grain

**GROUP BY result:** one row per department.

**Window result:** one row per employee.

This difference is the bridge to the deeper window-function work in Module 11.


# Part X · Ranking Within a Group

Use `PARTITION BY` to restart a window calculation for each department.


In [ ]:
display_query("""
SELECT
    name,
    department,
    salary,
    RANK() OVER (
        PARTITION BY department
        ORDER BY salary DESC
    ) AS department_rank
FROM employees
ORDER BY department, department_rank, name;
""", title="Salary rank within department")


Read the window definition in three parts:

1. `RANK()` — what calculation?
2. `PARTITION BY department` — where should it restart?
3. `ORDER BY salary DESC` — what order defines the ranking?


# Part XI · Compare Local and Global Averages

A window can show different levels of context on the same detail row.


In [ ]:
display_query("""
SELECT
    name,
    department,
    salary,
    ROUND(
        AVG(salary) OVER (
            PARTITION BY department
        ),
        2
    ) AS department_average,
    ROUND(
        AVG(salary) OVER (),
        2
    ) AS company_average
FROM employees
ORDER BY department, salary DESC;
""", title="Employee, department average, company average")


# Part XII · Running Totals

A running total keeps the current row but accumulates values in an order.

Example:

| row | revenue | running total |
|---:|---:|---:|
| 1 | 100 | 100 |
| 2 | 250 | 350 |
| 3 | 75 | 425 |


## 26. Running sales revenue


In [ ]:
display_query("""
SELECT
    sale_date,
    product,
    quantity * price AS row_revenue,
    SUM(quantity * price) OVER (
        ORDER BY sale_date, sale_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_revenue
FROM sales
ORDER BY sale_date, sale_id;
""", title="Running revenue")


The frame:

`ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`

means to include all ordered rows from the first row through the current row.

Module 11 will explore window frames in much more detail.


# Part XIII · Validation

A grouped query can be syntactically correct and still be analytically wrong.

Validation checks whether the summary agrees with the source.


## 27. Compare source rows with the number of diagnosis groups


In [ ]:
display_query("""
SELECT
    (SELECT COUNT(*) FROM admissions) AS source_rows,
    (
        SELECT COUNT(*)
        FROM (
            SELECT diagnosis
            FROM admissions
            GROUP BY diagnosis
        ) AS g
    ) AS diagnosis_groups;
""", title="Row-count validation")


## 28. Confirm that grouped revenue still equals source revenue


In [ ]:
display_query("""
WITH source AS (
    SELECT SUM(quantity * price) AS source_revenue
    FROM sales
),
grouped AS (
    SELECT
        category,
        SUM(quantity * price) AS category_revenue
    FROM sales
    GROUP BY category
)
SELECT
    (SELECT source_revenue FROM source) AS source_revenue,
    SUM(category_revenue) AS grouped_revenue,
    (SELECT source_revenue FROM source) = SUM(category_revenue) AS totals_match
FROM grouped;
""", title="Revenue validation")


### Useful validation questions

- Did the row count change as expected?
- Do grouped totals equal source totals?
- Did NULL values affect the metric?
- Did a join duplicate rows before aggregation?
- Are all expected categories present?
- Does the output grain match the original question?


# Part XIV · Applied Grade Example

The `grades` table gives us a larger grouping example.

Source grain:

> **one student-course-semester grade**


## 29. One row per course-semester


In [ ]:
display_query("""
SELECT
    course,
    semester,
    COUNT(*) AS students,
    ROUND(AVG(grade), 2) AS avg_grade,
    MIN(grade) AS min_grade,
    MAX(grade) AS max_grade
FROM grades
GROUP BY course, semester
ORDER BY semester, course;
""", title="Course-semester grade summary")


The output grain is:

> **one row per course-semester**

Both grouped columns together define the grain.


# Part XV · Guided Practice

Before each query, complete this sentence:

> **I want one output row per ________.**


## Practice 1 · COUNT

Return:

- all admissions
- completed admissions
- unique patients

<details>
<summary>Hint</summary>

Use `COUNT(*)`, `COUNT(discharge_time)`, and `COUNT(DISTINCT subject_id)`.

</details>


In [ ]:
practice_1 = """
SELECT
    -- count all rows,
    -- count completed rows,
    -- count unique patients
FROM admissions;
"""

# display_query(practice_1, title="Practice 1")


## Practice 2 · GROUP BY

Return one row per diagnosis with its admission count. Sort the most frequent diagnosis first.


In [ ]:
practice_2 = """
SELECT
    -- grouping column,
    -- aggregate
FROM admissions
GROUP BY
    -- grouping column
ORDER BY
    -- largest count first
;
"""


## Practice 3 · WHERE and HAVING

For sales rows priced at least $25:

- group by category
- calculate total units
- keep only categories with more than 10 units

<details>
<summary>Think before coding</summary>

Which condition belongs to individual rows?  
Which condition belongs to the grouped result?

</details>


In [ ]:
practice_3 = """
SELECT
    category,
    -- aggregate
FROM sales
WHERE
    -- row-level condition
GROUP BY category
HAVING
    -- group-level condition
;
"""


## Practice 4 · Conditional aggregation

Return one row per subject with separate Pneumonia and Fracture counts. Try `FILTER` first.


In [ ]:
practice_4 = """
SELECT
    subject_id,
    -- pneumonia count,
    -- fracture count
FROM admissions
GROUP BY subject_id
ORDER BY subject_id;
"""


## Practice 5 · Window ranking

Rank employees by salary inside each department without collapsing employee rows.


In [ ]:
practice_5 = """
SELECT
    name,
    department,
    salary,
    -- ranking window
FROM employees
ORDER BY department, salary DESC;
"""


## Practice 6 · Grades

Create one row per course with:

- number of grades
- average grade
- minimum grade
- maximum grade

Then extend it to one row per course-semester.


In [ ]:
practice_6 = """
SELECT
    course,
    -- aggregates
FROM grades
GROUP BY course
ORDER BY course;
"""


# Part XVI · Final Challenge

Build a sales dashboard with **one row per category**.

Return:

- category
- number of sales rows
- total units
- total revenue
- average price
- highest price

Then validate that the sum of category revenue equals the source revenue.


In [ ]:
final_query = """
SELECT
    category,
    -- number of sales rows,
    -- total units,
    -- total revenue,
    -- average price,
    -- highest price
FROM sales
GROUP BY category
ORDER BY category;
"""

# display_query(final_query, title="Final challenge")


# Quick Concept Check

### What does GROUP BY change?
<details><summary>Show answer</summary>It changes the output grain by collapsing detailed rows into groups.</details>

### What does COUNT(column) do with NULL?
<details><summary>Show answer</summary>It ignores NULL values.</details>

### Which clause filters source rows before grouping?
<details><summary>Show answer</summary>`WHERE`</details>

### Which clause filters aggregated groups?
<details><summary>Show answer</summary>`HAVING`</details>

### What is the key difference between GROUP BY and a window function?
<details><summary>Show answer</summary>GROUP BY collapses detail rows; a window function can keep detail rows while adding group-level calculations.</details>

### When is ROLLUP useful?
<details><summary>Show answer</summary>When you need hierarchical subtotals and a grand total.</details>

### Why validate grouped revenue?
<details><summary>Show answer</summary>To check that aggregation did not lose, duplicate, or misclassify source values.</details>


# Module 4 Summary

The key idea is:

> **Aggregation changes grain.**

Use this workflow:

1. Define the desired output grain.
2. Inspect the source rows.
3. Choose aggregate functions.
4. Add `GROUP BY` when multiple groups are needed.
5. Use `WHERE` for row-level filtering.
6. Use `HAVING` for group-level filtering.
7. Add conditional metrics when needed.
8. Use advanced grouping only when the report requires it.
9. Use windows when you want group context without collapsing rows.
10. Validate totals and row counts.

## Next
**Module 5: Reasoning with SQL Logic**
